In [1]:
import torch
import torchvision
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss
from sklearn.metrics import f1_score as sklearn_f1_score
from sklearn.metrics import precision_score, recall_score, accuracy_score

import torch.nn as nn
import torch.optim as optim
import copy

import tqdm

import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold

import pandas as pd
import matplotlib.pyplot as plt

from foldrm import Classifier
from utils import split_data # Or your stratified version if you prefer
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU found")

GPU available: False
No GPU found


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
random.seed(42)

# HUMAN THREATS Included

In [5]:
model, data = final_extinctionrisk_dataframe()

categorical_data = data.drop(model.numeric, axis=1)
categorical_data_without_y = categorical_data.drop(model.label, axis=1)
categorical_data_without_y_dummies = pd.get_dummies(categorical_data_without_y, dtype=int)
one_hot_dataset = pd.concat([data[model.numeric], categorical_data_without_y_dummies],axis=1)

X = one_hot_dataset

X = X.to_numpy()
mapping = {'Lower_risk':0, 'Higher_risk':1}
y = data[model.label]
y = y.map(mapping)
y = y.to_numpy()

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X.to(device)
y.to(device)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [6]:
def model_train(model, X_train, y_train, X_val, y_val):
    # loss function and optimizer
    loss_fn = nn.BCELoss()  # binary cross entropy
    optimizer = optim.Adam(model.parameters(), lr=0.001)
 
    n_epochs = 100   # number of epochs to run
    batch_size = 16  # size of each batch
    batch_start = torch.arange(0, len(X_train), batch_size)
 
    # Hold the best model
    best_acc = - np.inf   # init to negative infinity
    best_rec = - np.inf   # init to negative infinity
    best_pre = - np.inf   # init to negative infinity
    best_f1 = - np.inf   # init to negative infinity
    best_weights = None
 
    for epoch in range(n_epochs):
        model.train()
        with tqdm.tqdm(batch_start, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            for start in bar:
                # take a batch
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size].unsqueeze(dim=1)
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                # update weights
                optimizer.step()
                # print progress
                acc = (y_pred.round() == y_batch).float().mean()
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
        # evaluate accuracy at end of each epoch
        model.eval()
        y_pred = model(X_val)
        preds_cpu = y_pred.cpu()
        labels_cpu = y_val.cpu()
        preds_numpy = np.round(preds_cpu.detach().numpy().flatten())
        labels_numpy = labels_cpu.detach().numpy()
        f1 = sklearn_f1_score(labels_numpy, preds_numpy, average=None)
        acc = accuracy_score(labels_numpy, preds_numpy)
        precision = precision_score(labels_numpy, preds_numpy, average=None)
        recall = recall_score(labels_numpy, preds_numpy, average=None)
        
        if acc > best_acc:
            best_acc = acc
            best_weights = copy.deepcopy(model.state_dict())
            best_rec = recall
            best_pre = precision
            best_f1 = f1
            stop_state = epoch
    # restore model and return best accuracy
    model.load_state_dict(best_weights)
    return best_acc, best_rec, best_pre, best_f1, stop_state

In [7]:
class SLP(torch.nn.Module):
    def __init__(self, input_size):
        super(SLP, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        return x
    
# create model, train, and get accuracy
model = SLP(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Single Layer Perceptron, Including Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

C:\Users\anmarch\AppData\Local\Temp\ipykernel_64256\3631321629.py:36: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  loss=float(loss),
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

Single Layer Perceptron, Including Human Threats
Accuracy:0.8520154610712314
Precision:[0.86864932 0.71573604]
Recall[0.96159122 0.39943343]
F1:[0.91276042 0.51272727]
epoch:4


c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
class TwoLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(TwoLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 10)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(10, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.linear1(x))
        x = self.sigmoid(self.linear2(x))
        return x

# create model, train, and get accuracy
model = TwoLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("2 Layer MLP, Including Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


2 Layer MLP, Including Human Threats
Accuracy:0.8658199889563777
Precision:[0.87616099 0.78061224]
Recall[0.97050754 0.43342776]
F1:[0.92092418 0.55737705]
epoch:96


In [9]:
class FiveLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(FiveLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 10)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(10, 10)
        self.linear3 = torch.nn.Linear(10, 10)
        self.linear4 = torch.nn.Linear(10, 10)
        self.linear5 = torch.nn.Linear(10, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.linear1(x))
        x = self.activation(self.linear2(x))
        x = self.activation(self.linear3(x))
        x = self.activation(self.linear4(x))
        x = self.sigmoid(self.linear5(x))
        return x

# create model, train, and get accuracy
model = FiveLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("5 Layer MLP, Including Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

5 Layer MLP, Including Human Threats
Accuracy:0.8674765323025952
Precision:[0.87043796 0.83832335]
Recall[0.98148148 0.39660057]
F1:[0.92263056 0.53846154]
epoch:99


In [10]:
const_bnn_prior_parameters = {
        "prior_mu": 0.0,
        "prior_sigma": 1.0,
        "posterior_mu_init": 0.0,
        "posterior_rho_init": -3.0,
        "type": "Reparameterization",  # Flipout or Reparameterization
        "moped_enable": True,  # True to initialize mu/sigma from the pretrained dnn weights
        "moped_delta": 0.5,
}

model = TwoLayerMLPwithRELU(np.shape(X)[1])

dnn_to_bnn(model, const_bnn_prior_parameters)
model.to(device)

model = model
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

Accuracy:0.8558807288790723
Precision:[0.87150838 0.73      ]
Recall[0.96296296 0.41359773]
F1:[0.91495601 0.52802893]
epoch:93


# HUMAN THREATS not included

In [11]:
model, data = final_extinctionrisk_noth_dataframe()

categorical_data = data.drop(model.numeric, axis=1)
categorical_data_without_y = categorical_data.drop(model.label, axis=1)
categorical_data_without_y_dummies = pd.get_dummies(categorical_data_without_y, dtype=int)
one_hot_dataset = pd.concat([data[model.numeric], categorical_data_without_y_dummies],axis=1)

X = one_hot_dataset

X = X.to_numpy()
mapping = {'Lower_risk':0, 'Higher_risk':1}
y = data[model.label]
y = y.map(mapping)
y = y.to_numpy()

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X.to(device)
y.to(device)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [12]:
class SLP(torch.nn.Module):
    def __init__(self, input_size):
        super(SLP, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        return x
    
# create model, train, and get accuracy
model = SLP(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("Single Layer Perceptron, Without Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

Single Layer Perceptron, Without Human Threats
Accuracy:0.8564329099944782
Precision:[0.86884236 0.7486631 ]
Recall[0.96776406 0.39660057]
F1:[0.9156392  0.51851852]
epoch:13


c:\Users\anmarch\.conda\envs\bayesian-confold\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [13]:
class TwoLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(TwoLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 10)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(10, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.linear1(x))
        x = self.sigmoid(self.linear2(x))
        return x

# create model, train, and get accuracy
model = TwoLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("2 Layer MLP, Without Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

2 Layer MLP, Without Human Threats
Accuracy:0.8569850911098841
Precision:[0.87398628 0.72596154]
Recall[0.96090535 0.42776204]
F1:[0.91538713 0.53832442]
epoch:9


In [14]:
class FiveLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(FiveLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 10)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(10, 10)
        self.linear3 = torch.nn.Linear(10, 10)
        self.linear4 = torch.nn.Linear(10, 10)
        self.linear5 = torch.nn.Linear(10, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.linear1(x))
        x = self.activation(self.linear2(x))
        x = self.activation(self.linear3(x))
        x = self.activation(self.linear4(x))
        x = self.sigmoid(self.linear5(x))
        return x

# create model, train, and get accuracy
model = FiveLayerMLPwithRELU(np.shape(X)[1])
model.to(device)
acc, recall, precision, f1, epoch_num = model_train(model, X_train, y_train, X_test, y_test)
print("5 Layer MLP, Without Human Threats")
print("Accuracy:"+str(acc))
print("Precision:"+str(precision))
print("Recall"+str(recall))
print("F1:"+str(f1))
print("epoch:"+str(epoch_num))

5 Layer MLP, Without Human Threats
Accuracy:0.8575372722252899
Precision:[0.87313433 0.73399015]
Recall[0.96296296 0.42209632]
F1:[0.91585127 0.53597122]
epoch:70
